In [1]:
# !pip install pandas transformers torch scikit-learn -q
# !pip install ipywidgets -q

In [8]:
import pandas as pd
from transformers import pipeline
from sklearn.metrics import accuracy_score, classification_report
import re

In [9]:
support_tickets = pd.read_csv("customer_support_tickets.csv")

print("Dataset Shape:")
print(support_tickets.shape)

print("\nColumns:")
print(support_tickets.columns)

support_tickets.head()

Dataset Shape:
(8469, 17)

Columns:
Index(['Ticket ID', 'Customer Name', 'Customer Email', 'Customer Age',
       'Customer Gender', 'Product Purchased', 'Date of Purchase',
       'Ticket Type', 'Ticket Subject', 'Ticket Description', 'Ticket Status',
       'Resolution', 'Ticket Priority', 'Ticket Channel',
       'First Response Time', 'Time to Resolution',
       'Customer Satisfaction Rating'],
      dtype='object')


,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


In [10]:
support_tickets[
    ["Ticket Type", "Ticket Subject", "Ticket Description"]
].head()

,Ticket Type,Ticket Subject,Ticket Description
0,Technical issue,Product setup,I'm having an issue with the {product_purchase...
1,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...
2,Technical issue,Network problem,I'm facing a problem with my {product_purchase...
3,Billing inquiry,Account access,I'm having an issue with the {product_purchase...
4,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...


In [11]:
support_tickets[
    ["Ticket Type", "Ticket Subject", "Ticket Description"]
].isnull().sum()

Ticket Type           0
Ticket Subject        0
Ticket Description    0
dtype: int64

In [12]:
support_tickets["Ticket Type"].value_counts()

Ticket Type
Refund request          1752
Technical issue         1747
Cancellation request    1695
Product inquiry         1641
Billing inquiry         1634
Name: count, dtype: int64

In [ ]:
# support_tickets["clean_description"] = (
#     support_tickets["Ticket Description"]
#     .astype(str)
#     .apply(
#         lambda text: re.sub(r"\{.*?\}", "", text)
#     )
# )

In [ ]:
# support_tickets["ticket_text"] = (
#     support_tickets["Ticket Subject"].astype(str)
#     + ". "
#     + support_tickets["clean_description"]
# )

In [13]:
support_tickets["ticket_text"] = (
    support_tickets["Ticket Subject"].astype(str)
    + ". "
    + support_tickets["Ticket Description"].astype(str)
)

In [14]:
support_tickets[
    ["Ticket Type", "ticket_text"]
].head()

,Ticket Type,ticket_text
0,Technical issue,Product setup. I'm having an issue with the {p...
1,Technical issue,Peripheral compatibility. I'm having an issue ...
2,Technical issue,Network problem. I'm facing a problem with my ...
3,Billing inquiry,Account access. I'm having an issue with the {...
4,Billing inquiry,Data loss. I'm having an issue with the {produ...


In [16]:
support_tickets[
    ["Ticket Type", "ticket_text"]
].head()

,Ticket Type,ticket_text
0,Technical issue,Product setup. I'm having an issue with the {p...
1,Technical issue,Peripheral compatibility. I'm having an issue ...
2,Technical issue,Network problem. I'm facing a problem with my ...
3,Billing inquiry,Account access. I'm having an issue with the {...
4,Billing inquiry,Data loss. I'm having an issue with the {produ...


In [17]:
ticket_categories = [
    "Refund request",
    "Technical issue",
    "Cancellation request",
    "Product inquiry",
    "Billing inquiry"
]

In [18]:
classifier = pipeline(
    "zero-shot-classification",
    model="typeform/distilbert-base-uncased-mnli"
)

config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

c:\Users\786\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\786\.cache\huggingface\hub\models--typeform--distilbert-base-uncased-mnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/258 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [11]:
ticket_text = support_tickets["ticket_text"][0]

classification_result = classifier(
    ticket_text,
    ticket_categories
)

print("Ticket:\n")
print(ticket_text)

print("\nTop 3 Predicted Tags:\n")

for rank, (label, score) in enumerate(
    zip(
        classification_result["labels"][:3],
        classification_result["scores"][:3]
    ),
    start=1
):
    print(f"{rank}. {label} ({score:.2f})")

Ticket:

Product setup. I'm having an issue with the . Please assist.

Your billing zip code is: 71701.

We appreciate that you have requested a website address.

Please double check your email address. I've tried troubleshooting steps mentioned in the user manual, but the issue persists.

Top 3 Predicted Tags:

1. Billing inquiry (0.31)
2. Product inquiry (0.31)
3. Technical issue (0.19)


In [12]:
few_shot_categories = [
    "Technical issue: product not working, bug, error, troubleshooting problem",
    "Billing inquiry: payment issue, invoice problem, billing address, charges",
    "Cancellation request: cancel subscription, terminate account, stop service",
    "Product inquiry: product details, features, compatibility, information",
    "Refund request: refund money, reimbursement, return request"
]

In [13]:
few_shot_result = classifier(
    ticket_text,
    few_shot_categories
)

print("Top 3 Few-Shot Tags:\n")

for rank, (label, score) in enumerate(
    zip(
        few_shot_result["labels"][:3],
        few_shot_result["scores"][:3]
    ),
    start=1
):
    print(
        f"{rank}. {label.split(':')[0]} ({score:.2f})"
    )

Top 3 Few-Shot Tags:

1. Technical issue (0.37)
2. Billing inquiry (0.23)
3. Product inquiry (0.22)


In [23]:
evaluation_tickets = support_tickets.head(100).copy()

In [24]:
actual_ticket_types = (
    evaluation_tickets["Ticket Type"]
    .tolist()
)

In [25]:
predicted_ticket_types = []

for ticket_text in evaluation_tickets["ticket_text"]:

    classification_result = classifier(
        ticket_text,
        ticket_categories
    )

    predicted_ticket_types.append(
        classification_result["labels"][0]
    )

In [26]:
accuracy = accuracy_score(
    actual_ticket_types,
    predicted_ticket_types
)

print("Accuracy:", accuracy)

Accuracy: 0.2


In [27]:
print(
    classification_report(
        actual_ticket_types,
        predicted_ticket_types,
        zero_division=0
    )
)

                      precision    recall  f1-score   support

     Billing inquiry       0.00      0.00      0.00        17
Cancellation request       0.40      0.12      0.18        17
     Product inquiry       0.20      0.71      0.31        21
      Refund request       0.11      0.06      0.07        18
     Technical issue       0.20      0.07      0.11        27

            accuracy                           0.20       100
           macro avg       0.18      0.19      0.14       100
        weighted avg       0.18      0.20      0.14       100



In [20]:
comparison_table = pd.DataFrame({
    "Method": [
        "Zero-Shot",
        "Few-Shot"
    ],
    "Observation": [
        "Uses category names only",
        "Uses category descriptions/examples"
    ]
})

comparison_table

,Method,Observation
0,Zero-Shot,Uses category names only
1,Few-Shot,Uses category descriptions/examples
